# Data Preparation

## Phase A — Incident sample preparation
### A1. Imports and configuration

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path("data/nsw2025")
DATA_DIR = ROOT / "new data"
OUTPUT_DIR = Path("prepared_data_classification")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLOW_FILE = DATA_DIR / "final_hourly_flow_allfeature_with_timefeat.csv"
INCIDENT_FILE = DATA_DIR / "final_incidents_with_weather.csv"

print("Flow file:", FLOW_FILE)
print("Incident file:", INCIDENT_FILE)

Flow file: data\nsw2025\new data\final_hourly_flow_allfeature_with_timefeat.csv
Incident file: data\nsw2025\new data\final_incidents_with_weather.csv


### A2. Load the two source datasets

In [7]:
flow_df = pd.read_csv(
    FLOW_FILE,
    dtype={
        "station_id": "string",
        "incident_id": "string",
    },
    low_memory=False,
)

incident_df = pd.read_csv(
    INCIDENT_FILE,
    dtype={
        "station_id": "string",
        "incident_id": "string",
    },
    low_memory=False,
)

print("Hourly flow shape:", flow_df.shape)
print("Incident shape:", incident_df.shape)

Hourly flow shape: (1007400, 31)
Incident shape: (3197, 26)


In [8]:
display(flow_df.head())
display(incident_df.head())

print("\nHourly flow columns:")
print(flow_df.columns.tolist())

print("\nIncident columns:")
print(incident_df.columns.tolist())

,station_id,timestamp,hour_sin,hour_cos,day_of_week,is_weekend,is_holiday,hour,total_flow,wgs84_latitude,...,is_major_incident,impact_sequence_hour,precipitation,weather_code,apparent_temperature,temperature_2m,wind_gusts_10m,relative_humidity,incident_count,is_anomaly
0,100001,2025-01-01 00:00:00,0.000000,1.000000,2,0,1,0,13.0,-33.878181,...,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
1,100001,2025-01-01 01:00:00,0.258819,0.965926,2,0,1,1,9.0,-33.878181,...,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
2,100001,2025-01-01 02:00:00,0.500000,0.866025,2,0,1,2,10.0,-33.878181,...,0,-1.0,0.0,1.0,23.952227,21.00,5.040000,86.98510,0,0
3,100001,2025-01-01 03:00:00,0.707107,0.707107,2,0,1,3,6.0,-33.878181,...,0,-1.0,0.0,1.0,23.543003,20.65,5.760000,88.87851,0,0
4,100001,2025-01-01 04:00:00,0.866025,0.500000,2,0,1,4,7.0,-33.878181,...,0,-1.0,0.0,1.0,23.117048,20.25,6.479999,90.81496,0,0


,station_id,Abs PM_x,Abs PM_y,incident_id,dis,hazard_type,incident_kind,match_hour,calculated_duration_hours,impact_sequence_hour,...,traffic_volume_desc,affected_direction,temperature_2m,rain,precipitation,weather_code,apparent_temperature,relative_humidity,wind_gusts_10m,dew_point_2m
0,F3FWY003,-33.560299,151.192551,219468-webtirf,0.104,CRASH,Unplanned,2025-01-02 01:00:00,1.541561,0,...,Heavy,Southbound,21.25,0.0,0.0,1.0,22.991776,92.586970,28.440000,20.00
1,F3FWY003,-33.560299,151.192551,219468-webtirf,0.104,CRASH,Unplanned,2025-01-02 02:00:00,1.541561,1,...,Heavy,Southbound,21.35,0.0,0.0,2.0,22.738764,92.021120,36.360000,20.00
2,7251,-33.874916,151.155319,219979-webtirf,0.073,BREAKDOWN,Unplanned,2025-01-08 23:00:00,0.500000,0,...,NaN,Westbound,18.70,1.4,1.4,61.0,17.486103,89.003174,52.199997,16.85
3,7179,-33.817272,150.962784,220424-webtirf,0.285,CRASH,Unplanned,2025-01-13 21:00:00,0.500000,0,...,NaN,Southbound,24.20,0.0,0.0,3.0,28.105839,84.389980,11.159999,21.40
4,7120,-33.886311,151.217911,220471-webtirf,0.100,BREAKDOWN,Unplanned,2025-01-14 02:00:00,0.500000,0,...,NaN,Northbound,21.95,0.0,0.0,3.0,25.808100,95.808860,12.959999,21.25



Hourly flow columns:
['station_id', 'timestamp', 'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'is_holiday', 'hour', 'total_flow', 'wgs84_latitude', 'wgs84_longitude', 'road_name', 'suburb', 'post_code', 'device_type', 'quality_rating', 'lane_count', 'road_functional_hierarchy', 'distance_to_intersection', 'incident_id', 'incident_type', 'is_major_incident', 'impact_sequence_hour', 'precipitation', 'weather_code', 'apparent_temperature', 'temperature_2m', 'wind_gusts_10m', 'relative_humidity', 'incident_count', 'is_anomaly']

Incident columns:
['station_id', 'Abs PM_x', 'Abs PM_y', 'incident_id', 'dis', 'hazard_type', 'incident_kind', 'match_hour', 'calculated_duration_hours', 'impact_sequence_hour', 'is_major_incident', 'is_local_road', 'speed_limit', 'advice_a', 'advice_b', 'queue_length_km', 'traffic_volume_desc', 'affected_direction', 'temperature_2m', 'rain', 'precipitation', 'weather_code', 'apparent_temperature', 'relative_humidity', 'wind_gusts_10m', 'dew_point_2m']


### A3. Canonicalize station IDs

In [9]:
def canonicalize_station_id(series):
    """
    Standardize station identifiers without converting them to numeric.
    """
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
    )


flow_df["station_id"] = canonicalize_station_id(
    flow_df["station_id"]
)

incident_df["station_id"] = canonicalize_station_id(
    incident_df["station_id"]
)

print("Flow stations:", flow_df["station_id"].nunique())
print("Incident stations:", incident_df["station_id"].nunique())

Flow stations: 115
Incident stations: 67


### A4. Parse timestamps

In [10]:
flow_df["timestamp"] = pd.to_datetime(
    flow_df["timestamp"],
    errors="coerce",
)

incident_df["match_hour"] = pd.to_datetime(
    incident_df["match_hour"],
    errors="coerce",
)

print(
    "Invalid flow timestamps:",
    flow_df["timestamp"].isna().sum()
)

print(
    "Invalid incident timestamps:",
    incident_df["match_hour"].isna().sum()
)

Invalid flow timestamps: 0
Invalid incident timestamps: 0


In [11]:
flow_non_hourly = flow_df[
    (flow_df["timestamp"].dt.minute != 0)
    | (flow_df["timestamp"].dt.second != 0)
]

incident_non_hourly = incident_df[
    (incident_df["match_hour"].dt.minute != 0)
    | (incident_df["match_hour"].dt.second != 0)
]

print(
    "Non-hour-aligned flow timestamps:",
    len(flow_non_hourly)
)

print(
    "Non-hour-aligned incident timestamps:",
    len(incident_non_hourly)
)

Non-hour-aligned flow timestamps: 0
Non-hour-aligned incident timestamps: 0


### A5. Validate total_flow

In [12]:
flow_df["total_flow"] = pd.to_numeric(
    flow_df["total_flow"],
    errors="coerce",
)

# Negative traffic counts are invalid
negative_flow_mask = flow_df["total_flow"] < 0

print(
    "Negative total_flow values:",
    negative_flow_mask.sum()
)

flow_df.loc[
    negative_flow_mask,
    "total_flow"
] = np.nan

print(
    "Missing total_flow values:",
    flow_df["total_flow"].isna().sum()
)

Negative total_flow values: 0
Missing total_flow values: 0


### A6. Check duplicate station-hour records

In [13]:
duplicate_station_hours = flow_df[
    flow_df.duplicated(
        subset=["station_id", "timestamp"],
        keep=False,
    )
].sort_values(
    ["station_id", "timestamp"]
)

print(
    "Duplicate station-hour rows:",
    len(duplicate_station_hours)
)

Duplicate station-hour rows: 0


In [14]:
assert not flow_df.duplicated(
    subset=["station_id", "timestamp"]
).any(), (
    "Duplicate station-hour observations exist."
)

print(
    "PASS: each station-hour combination is unique."
)

PASS: each station-hour combination is unique.


### A7. Check that incident stations exist in the traffic data

In [15]:
flow_stations = set(
    flow_df["station_id"].dropna().unique()
)

incident_stations = set(
    incident_df["station_id"].dropna().unique()
)

matched_stations = (
    flow_stations
    & incident_stations
)

missing_incident_stations = (
    incident_stations
    - flow_stations
)

print(
    "Flow stations:",
    len(flow_stations)
)

print(
    "Incident stations:",
    len(incident_stations)
)

print(
    "Matched incident stations:",
    len(matched_stations)
)

print(
    "Incident stations missing from flow:",
    len(missing_incident_stations)
)

Flow stations: 115
Incident stations: 67
Matched incident stations: 67
Incident stations missing from flow: 0


### A8. Canonicalize incident labels

In [16]:
def canonicalize_incident_label(value):
    """
    Convert incident labels to uppercase underscore format.

    Example:
        'Special Event' -> 'SPECIAL_EVENT'
        'Traffic Lights Blacked Out'
        -> 'TRAFFIC_LIGHTS_BLACKED_OUT'
    """

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    value = re.sub(
        r"[\s\-/]+",
        "_",
        value,
    )

    value = re.sub(
        r"[^A-Z0-9_]",
        "_",
        value,
    )

    value = re.sub(
        r"_+",
        "_",
        value,
    )

    return value.strip("_")


incident_df[
    "hazard_type_canonical"
] = incident_df[
    "hazard_type"
].apply(
    canonicalize_incident_label
)

In [17]:
label_counts = (
    incident_df[
        "hazard_type_canonical"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    label_counts.to_frame("rows")
)

,rows
hazard_type_canonical,
BREAKDOWN,1136
CRASH,688
SPECIAL_EVENT,576
SCHEDULED_ROADWORK,145
HAZARD,144
TRAFFIC_LIGHTS_BLACKED_OUT,138
EMERGENCY_ROADWORK,103
CHANGED_TRAFFIC_CONDITIONS,86
ADVERSE_WEATHER,73


### A9. Apply the locked three-class taxonomy

In [18]:
ACCIDENT_LABELS = {
    "CRASH",
}


OTHER_DISRUPTION_LABELS = {
    "BREAKDOWN",
    "HAZARD",
    "ADVERSE_WEATHER",
    "BURST_WATER_MAIN",
    "CHANGED_TRAFFIC_CONDITIONS",
    "TRAFFIC_LIGHTS_FLASHING_YELLOW",
    "TRAFFIC_LIGHTS_BLACKED_OUT",
    "HOLIDAY_TRAFFIC",
    "SCHEDULED_ROADWORK",
    "SPECIAL_EVENT",
}


def map_target_class(label):

    if pd.isna(label):
        return "EXCLUDED_UNKNOWN"

    if label in ACCIDENT_LABELS:
        return "ACCIDENT"

    if label in OTHER_DISRUPTION_LABELS:
        return "OTHER_DISRUPTION"

    return "EXCLUDED_UNKNOWN"


incident_df[
    "target_class"
] = incident_df[
    "hazard_type_canonical"
].apply(
    map_target_class
)

In [19]:
taxonomy_audit = (
    incident_df
    .groupby(
        [
            "hazard_type_canonical",
            "target_class",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="row_count"
    )
    .sort_values(
        [
            "target_class",
            "row_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

display(taxonomy_audit)

,hazard_type_canonical,target_class,row_count
6,CRASH,ACCIDENT,688
7,EMERGENCY_ROADWORK,EXCLUDED_UNKNOWN,103
2,BUILDING_FIRE,EXCLUDED_UNKNOWN,16
8,FLOODING,EXCLUDED_UNKNOWN,10
4,BUSHFIRE,EXCLUDED_UNKNOWN,4
11,HEAVY_TRAFFIC,EXCLUDED_UNKNOWN,2
9,GRASS_FIRE,EXCLUDED_UNKNOWN,1
13,LATE_FINISHING_ROADWORK,EXCLUDED_UNKNOWN,1
1,BREAKDOWN,OTHER_DISRUPTION,1136
15,SPECIAL_EVENT,OTHER_DISRUPTION,576


In [20]:
print(
    incident_df[
        "target_class"
    ].value_counts()
)

target_class
OTHER_DISRUPTION    2372
ACCIDENT             688
EXCLUDED_UNKNOWN     137
Name: count, dtype: int64


### A10. Create one incident sample per incident × station

In [ ]:
# Ensure IDs remain strings.
incident_df[
    "incident_id"
] = (
    incident_df[
        "incident_id"
    ]
    .astype("string")
    .str.strip()
)

In [22]:
# Construct the event-level incident table.
incident_samples = (
    incident_df
    .sort_values(
        [
            "incident_id",
            "station_id",
            "match_hour",
        ]
    )
    .groupby(
        [
            "incident_id",
            "station_id",
        ],
        as_index=False,
    )
    .agg(
        anchor_time=(
            "match_hour",
            "min",
        ),

        original_hazard_type=(
            "hazard_type",
            "first",
        ),

        hazard_type_canonical=(
            "hazard_type_canonical",
            "first",
        ),

        target_class=(
            "target_class",
            "first",
        ),

        calculated_duration_hours=(
            "calculated_duration_hours",
            "first",
        ),

        is_major_incident=(
            "is_major_incident",
            "first",
        ),

        source_row_count=(
            "match_hour",
            "size",
        ),
    )
)

In [23]:
# Create a unique modelling sample identifier.
incident_samples[
    "event_sample_id"
] = (
    "INCIDENT__"
    + incident_samples[
        "incident_id"
    ].astype(str)
    + "__"
    + incident_samples[
        "station_id"
    ].astype(str)
)

In [24]:
# Check uniqueness.
assert incident_samples[
    "event_sample_id"
].is_unique

print(
    "Unique incident IDs:",
    incident_df[
        "incident_id"
    ].nunique()
)

print(
    "Incident-station samples:",
    len(incident_samples)
)

Unique incident IDs: 1628
Incident-station samples: 1828


### A11. Exclude unknown taxonomy samples

In [25]:
exclusion_counts = (
    incident_samples[
        "target_class"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    exclusion_counts.to_frame(
        "incident_station_samples"
    )
)

,incident_station_samples
target_class,
OTHER_DISRUPTION,1232
ACCIDENT,556
EXCLUDED_UNKNOWN,40


In [ ]:
# retain the two incident classes.
valid_incident_samples = (
    incident_samples[
        incident_samples[
            "target_class"
        ].isin(
            [
                "ACCIDENT",
                "OTHER_DISRUPTION",
            ]
        )
    ]
    .copy()
)

excluded_incident_samples = (
    incident_samples[
        ~incident_samples[
            "target_class"
        ].isin(
            [
                "ACCIDENT",
                "OTHER_DISRUPTION",
            ]
        )
    ]
    .copy()
)

print(
    "Valid incident samples:",
    len(valid_incident_samples)
)

print(
    "Excluded incident samples:",
    len(excluded_incident_samples)
)

display(
    valid_incident_samples[
        "target_class"
    ]
    .value_counts()
)

Valid incident samples: 1788
Excluded incident samples: 40


target_class
OTHER_DISRUPTION    1232
ACCIDENT             556
Name: count, dtype: int64

### A12. Add the seven-hour window boundaries

In [27]:
valid_incident_samples[
    "window_start"
] = (
    valid_incident_samples[
        "anchor_time"
    ]
    - pd.Timedelta(
        hours=3
    )
)

valid_incident_samples[
    "window_end"
] = (
    valid_incident_samples[
        "anchor_time"
    ]
    + pd.Timedelta(
        hours=3
    )
)

In [29]:
# Store each expected timestep.
TIME_OFFSETS = {
    "t_minus_3": -3,
    "t_minus_2": -2,
    "t_minus_1": -1,
    "t0": 0,
    "t_plus_1": 1,
    "t_plus_2": 2,
    "t_plus_3": 3,
}


for column, offset in TIME_OFFSETS.items():

    valid_incident_samples[
        column
    ] = (
        valid_incident_samples[
            "anchor_time"
        ]
        + pd.Timedelta(
            hours=offset
        )
    )

In [30]:
assert valid_incident_samples["anchor_time"].notna().all()
assert valid_incident_samples["station_id"].notna().all()
assert valid_incident_samples["event_sample_id"].is_unique

print("PASS: incident sample table is valid.")

PASS: incident sample table is valid.


In [31]:
display(
    valid_incident_samples[
        [
            "event_sample_id",
            "incident_id",
            "station_id",
            "anchor_time",
            "target_class",
            "window_start",
            "window_end",
            "t_minus_3",
            "t_minus_2",
            "t_minus_1",
            "t0",
            "t_plus_1",
            "t_plus_2",
            "t_plus_3",
        ]
    ].head()
)

,event_sample_id,incident_id,station_id,anchor_time,target_class,window_start,window_end,t_minus_3,t_minus_2,t_minus_1,t0,t_plus_1,t_plus_2,t_plus_3
0,INCIDENT__219427-webtirf__7120,219427-webtirf,7120,2025-01-01 04:00:00,OTHER_DISRUPTION,2025-01-01 01:00:00,2025-01-01 07:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00,2025-01-01 05:00:00,2025-01-01 06:00:00,2025-01-01 07:00:00
1,INCIDENT__219427-webtirf__7121,219427-webtirf,7121,2025-01-01 04:00:00,OTHER_DISRUPTION,2025-01-01 01:00:00,2025-01-01 07:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00,2025-01-01 05:00:00,2025-01-01 06:00:00,2025-01-01 07:00:00
2,INCIDENT__219440.0-webtirf__7168,219440.0-webtirf,7168,2025-01-01 11:00:00,OTHER_DISRUPTION,2025-01-01 08:00:00,2025-01-01 14:00:00,2025-01-01 08:00:00,2025-01-01 09:00:00,2025-01-01 10:00:00,2025-01-01 11:00:00,2025-01-01 12:00:00,2025-01-01 13:00:00,2025-01-01 14:00:00
3,INCIDENT__219468-webtirf__F3FWY003,219468-webtirf,F3FWY003,2025-01-02 01:00:00,ACCIDENT,2025-01-01 22:00:00,2025-01-02 04:00:00,2025-01-01 22:00:00,2025-01-01 23:00:00,2025-01-02 00:00:00,2025-01-02 01:00:00,2025-01-02 02:00:00,2025-01-02 03:00:00,2025-01-02 04:00:00
4,INCIDENT__219481-webtirf__7179,219481-webtirf,7179,2025-01-02 03:00:00,OTHER_DISRUPTION,2025-01-02 00:00:00,2025-01-02 06:00:00,2025-01-02 00:00:00,2025-01-02 01:00:00,2025-01-02 02:00:00,2025-01-02 03:00:00,2025-01-02 04:00:00,2025-01-02 05:00:00,2025-01-02 06:00:00


### A13 — Extract 7-hour traffic windows

In [ ]:
# Prepare a lookup table
flow_lookup = (
    flow_df[
        [
            "station_id",
            "timestamp",
            "total_flow",
        ]
    ]
    .copy()
)

print("Flow lookup shape:", flow_lookup.shape)

Flow lookup shape: (1007400, 3)


In [33]:
# Convert incident samples to long format
TIME_OFFSETS = {
    "t_minus_3": -3,
    "t_minus_2": -2,
    "t_minus_1": -1,
    "t0": 0,
    "t_plus_1": 1,
    "t_plus_2": 2,
    "t_plus_3": 3,
}


window_rows = []

for timestep_name, offset in TIME_OFFSETS.items():

    temp = valid_incident_samples[
        [
            "event_sample_id",
            "incident_id",
            "station_id",
            "anchor_time",
            "target_class",
            "calculated_duration_hours",
        ]
    ].copy()

    temp["timestep"] = timestep_name
    temp["relative_hour"] = offset

    temp["timestamp"] = (
        temp["anchor_time"]
        + pd.to_timedelta(
            offset,
            unit="h",
        )
    )

    window_rows.append(temp)


incident_windows_long = pd.concat(
    window_rows,
    ignore_index=True,
)

print(
    "Long incident-window rows:",
    len(incident_windows_long)
)

Long incident-window rows: 12516


### A14 — Join traffic flow to every required timestamp

In [34]:
incident_windows_long = (
    incident_windows_long
    .merge(
        flow_lookup,
        on=[
            "station_id",
            "timestamp",
        ],
        how="left",
        validate="many_to_one",
    )
)

In [35]:
display(
    incident_windows_long.head(14)
)

,event_sample_id,incident_id,station_id,anchor_time,target_class,calculated_duration_hours,timestep,relative_hour,timestamp,total_flow
0,INCIDENT__219427-webtirf__7120,219427-webtirf,7120,2025-01-01 04:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-01 01:00:00,0.0
1,INCIDENT__219427-webtirf__7121,219427-webtirf,7121,2025-01-01 04:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-01 01:00:00,0.0
2,INCIDENT__219440.0-webtirf__7168,219440.0-webtirf,7168,2025-01-01 11:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-01 08:00:00,13.0
3,INCIDENT__219468-webtirf__F3FWY003,219468-webtirf,F3FWY003,2025-01-02 01:00:00,ACCIDENT,1.541561,t_minus_3,-3,2025-01-01 22:00:00,0.0
4,INCIDENT__219481-webtirf__7179,219481-webtirf,7179,2025-01-02 03:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-02 00:00:00,21.0
5,INCIDENT__219562-webtirf__F3FWY003,219562-webtirf,F3FWY003,2025-01-04 00:00:00,ACCIDENT,0.500000,t_minus_3,-3,2025-01-03 21:00:00,154.0
6,INCIDENT__219564-webtirf__33014,219564-webtirf,33014,2025-01-04 00:00:00,OTHER_DISRUPTION,0.582011,t_minus_3,-3,2025-01-03 21:00:00,26.0
7,INCIDENT__219585-webtirf__F3FWY003,219585-webtirf,F3FWY003,2025-01-04 06:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-04 03:00:00,158.0
8,INCIDENT__219601-webtirf__7123-PR,219601-webtirf,7123-PR,2025-01-04 19:00:00,OTHER_DISRUPTION,2.675307,t_minus_3,-3,2025-01-04 16:00:00,477.0
9,INCIDENT__219613-webtirf__22001,219613-webtirf,22001,2025-01-05 05:00:00,OTHER_DISRUPTION,0.500000,t_minus_3,-3,2025-01-05 02:00:00,117.0


### A15 — Check missing flow observations

In [36]:
missing_flow_summary = (
    incident_windows_long
    .groupby(
        "event_sample_id"
    )["total_flow"]
    .agg(
        missing_count=lambda x: x.isna().sum(),
        available_count=lambda x: x.notna().sum(),
    )
    .reset_index()
)

In [37]:
display(
    missing_flow_summary[
        "missing_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "missing_hours"
    )
    .to_frame(
        "samples"
    )
)

,samples
missing_hours,
0,1788


### A16 — Inspect incomplete samples

In [38]:
incomplete_samples = (
    missing_flow_summary[
        missing_flow_summary[
            "missing_count"
        ] > 0
    ]
    .copy()
)

print(
    "Samples with incomplete windows:",
    len(incomplete_samples)
)

Samples with incomplete windows: 0


In [39]:
display(
    incident_windows_long[
        incident_windows_long[
            "event_sample_id"
        ].isin(
            incomplete_samples[
                "event_sample_id"
            ].head(10)
        )
    ]
    .sort_values(
        [
            "event_sample_id",
            "relative_hour",
        ]
    )
)

,event_sample_id,incident_id,station_id,anchor_time,target_class,calculated_duration_hours,timestep,relative_hour,timestamp,total_flow


### A17 — Apply the locked missing-value rule

In [42]:
def interpolate_incident_window(group):
    """
    Apply strict within-segment interpolation.

    No extrapolation and no interpolation across
    the pre-event / response boundary.
    """

    group = (
        group
        .sort_values(
            "relative_hour"
        )
        .copy()
    )

    group[
        "flow_was_interpolated"
    ] = 0

    group[
        "window_valid"
    ] = True

    group[
        "exclusion_reason"
    ] = pd.NA

    missing_mask = (
        group[
            "total_flow"
        ].isna()
    )

    missing_count = (
        missing_mask.sum()
    )

    # No missing values
    if missing_count == 0:
        return group

    # More than one missing value
    if missing_count > 1:

        group[
            "window_valid"
        ] = False

        group[
            "exclusion_reason"
        ] = (
            "MORE_THAN_ONE_MISSING_FLOW"
        )

        return group

    # Exactly one missing observation
    missing_index = (
        group[
            missing_mask
        ].index[0]
    )

    missing_hour = (
        group.loc[
            missing_index,
            "relative_hour",
        ]
    )

    # --------------------------------------------
    # Define allowed segment
    # --------------------------------------------

    if missing_hour < 0:

        allowed_hours = [
            -3,
            -2,
            -1,
        ]

    else:

        allowed_hours = [
            0,
            1,
            2,
            3,
        ]

    segment = (
        group[
            group[
                "relative_hour"
            ].isin(
                allowed_hours
            )
        ]
        .copy()
        .sort_values(
            "relative_hour"
        )
    )

    # --------------------------------------------
    # Find observations immediately before/after
    # within the same segment
    # --------------------------------------------

    before = (
        segment[
            (
                segment[
                    "relative_hour"
                ]
                < missing_hour
            )
            &
            (
                segment[
                    "total_flow"
                ].notna()
            )
        ]
        .tail(1)
    )

    after = (
        segment[
            (
                segment[
                    "relative_hour"
                ]
                > missing_hour
            )
            &
            (
                segment[
                    "total_flow"
                ].notna()
            )
        ]
        .head(1)
    )

    # True interpolation requires both sides
    if (
        before.empty
        or after.empty
    ):

        group[
            "window_valid"
        ] = False

        group[
            "exclusion_reason"
        ] = (
            "MISSING_FLOW_CANNOT_BE_INTERPOLATED"
        )

        return group

    x1 = (
        before[
            "relative_hour"
        ].iloc[0]
    )

    y1 = (
        before[
            "total_flow"
        ].iloc[0]
    )

    x2 = (
        after[
            "relative_hour"
        ].iloc[0]
    )

    y2 = (
        after[
            "total_flow"
        ].iloc[0]
    )

    interpolated_value = (
        y1
        +
        (
            (missing_hour - x1)
            /
            (x2 - x1)
        )
        *
        (
            y2 - y1
        )
    )

    group.loc[
        missing_index,
        "total_flow",
    ] = interpolated_value

    group.loc[
        missing_index,
        "flow_was_interpolated",
    ] = 1

    return group

### A18 — Apply the missing-value processing

In [43]:
incident_windows_processed = (
    incident_windows_long
    .groupby(
        "event_sample_id",
        group_keys=False,
    )
    .apply(
        interpolate_incident_window
    )
    .reset_index(
        drop=True
    )
)

### A19 — Create the window-level audit

In [44]:
incident_window_audit = (
    incident_windows_processed
    .groupby(
        "event_sample_id"
    )
    .agg(
        station_id=(
            "station_id",
            "first",
        ),

        incident_id=(
            "incident_id",
            "first",
        ),

        anchor_time=(
            "anchor_time",
            "first",
        ),

        target_class=(
            "target_class",
            "first",
        ),

        window_valid=(
            "window_valid",
            "all",
        ),

        interpolated_points=(
            "flow_was_interpolated",
            "sum",
        ),

        exclusion_reason=(
            "exclusion_reason",
            "first",
        ),

        final_missing_count=(
            "total_flow",
            lambda x: x.isna().sum(),
        ),
    )
    .reset_index()
)

In [45]:
print(
    "Total incident samples:",
    len(
        incident_window_audit
    )
)

print(
    "\nWindow validity:"
)

display(
    incident_window_audit[
        "window_valid"
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

print(
    "\nInterpolated points:"
)

display(
    incident_window_audit[
        "interpolated_points"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "samples"
    )
)

print(
    "\nExclusion reasons:"
)

display(
    incident_window_audit[
        "exclusion_reason"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "samples"
    )
)

Total incident samples: 1788

Window validity:


,samples
window_valid,
True,1788



Interpolated points:


,samples
interpolated_points,
0,1788



Exclusion reasons:


,samples
exclusion_reason,
None,1788


### A20 — Keep valid incident windows

In [46]:
valid_window_ids = (
    incident_window_audit.loc[
        incident_window_audit[
            "window_valid"
        ],
        "event_sample_id",
    ]
)


valid_incident_windows_long = (
    incident_windows_processed[
        incident_windows_processed[
            "event_sample_id"
        ].isin(
            valid_window_ids
        )
    ]
    .copy()
)

In [47]:
print(
    "Valid incident windows:",
    valid_incident_windows_long[
        "event_sample_id"
    ].nunique()
)

print(
    "Expected long rows:",
    valid_incident_windows_long[
        "event_sample_id"
    ].nunique()
    * 7
)

print(
    "Actual long rows:",
    len(
        valid_incident_windows_long
    )
)

Valid incident windows: 1788
Expected long rows: 12516
Actual long rows: 12516


### A21 — Convert to one row per incident sample

In [61]:
incident_flow_wide = (
    valid_incident_windows_long
    .pivot(
        index="event_sample_id",
        columns="timestep",
        values="total_flow",
    )
    .reset_index()
)

# Remove the pivot column-axis name
incident_flow_wide.columns.name = None

In [62]:
print(
    incident_flow_wide.columns.tolist()
)

['event_sample_id', 't0', 't_minus_1', 't_minus_2', 't_minus_3', 't_plus_1', 't_plus_2', 't_plus_3']


In [ ]:
# Rename flow columns clearly
FLOW_RENAME_MAP = {
    "t_minus_3": "flow_t_minus_3",
    "t_minus_2": "flow_t_minus_2",
    "t_minus_1": "flow_t_minus_1",
    "t0": "flow_t0",
    "t_plus_1": "flow_t_plus_1",
    "t_plus_2": "flow_t_plus_2",
    "t_plus_3": "flow_t_plus_3",
}


incident_flow_wide = (
    incident_flow_wide
    .rename(
        columns=FLOW_RENAME_MAP
    )
)

In [64]:
FLOW_COLUMNS = [
    "flow_t_minus_3",
    "flow_t_minus_2",
    "flow_t_minus_1",
    "flow_t0",
    "flow_t_plus_1",
    "flow_t_plus_2",
    "flow_t_plus_3",
]

In [65]:
display(
    incident_flow_wide.head()
)

,event_sample_id,flow_t0,flow_t_minus_1,flow_t_minus_2,flow_t_minus_3,flow_t_plus_1,flow_t_plus_2,flow_t_plus_3
0,INCIDENT__219427-webtirf__7120,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,INCIDENT__219427-webtirf__7121,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,INCIDENT__219440.0-webtirf__7168,10.0,13.0,4.0,13.0,16.0,12.0,7.0
3,INCIDENT__219468-webtirf__F3FWY003,19.0,31.0,0.0,0.0,30.0,27.0,0.0
4,INCIDENT__219481-webtirf__7179,34.0,33.0,30.0,21.0,73.0,105.0,160.0


In [66]:
valid_incident_dataset = (
    valid_incident_samples
    .merge(
        incident_flow_wide,
        on="event_sample_id",
        how="inner",
        validate="one_to_one",
    )
)

In [67]:
display(
    valid_incident_dataset[
        [
            "event_sample_id",
            "incident_id",
            "station_id",
            "anchor_time",
            "target_class",
            *FLOW_COLUMNS,
        ]
    ].head()
)

,event_sample_id,incident_id,station_id,anchor_time,target_class,flow_t_minus_3,flow_t_minus_2,flow_t_minus_1,flow_t0,flow_t_plus_1,flow_t_plus_2,flow_t_plus_3
0,INCIDENT__219427-webtirf__7120,219427-webtirf,7120,2025-01-01 04:00:00,OTHER_DISRUPTION,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,INCIDENT__219427-webtirf__7121,219427-webtirf,7121,2025-01-01 04:00:00,OTHER_DISRUPTION,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,INCIDENT__219440.0-webtirf__7168,219440.0-webtirf,7168,2025-01-01 11:00:00,OTHER_DISRUPTION,13.0,4.0,13.0,10.0,16.0,12.0,7.0
3,INCIDENT__219468-webtirf__F3FWY003,219468-webtirf,F3FWY003,2025-01-02 01:00:00,ACCIDENT,0.0,0.0,31.0,19.0,30.0,27.0,0.0
4,INCIDENT__219481-webtirf__7179,219481-webtirf,7179,2025-01-02 03:00:00,OTHER_DISRUPTION,21.0,30.0,33.0,34.0,73.0,105.0,160.0


In [68]:
display(
    valid_incident_dataset[
        [
            "event_sample_id",
            "station_id",
            "anchor_time",

            "t_minus_3",
            "flow_t_minus_3",

            "t_minus_2",
            "flow_t_minus_2",

            "t_minus_1",
            "flow_t_minus_1",

            "t0",
            "flow_t0",

            "t_plus_1",
            "flow_t_plus_1",

            "t_plus_2",
            "flow_t_plus_2",

            "t_plus_3",
            "flow_t_plus_3",
        ]
    ].head()
)

,event_sample_id,station_id,anchor_time,t_minus_3,flow_t_minus_3,t_minus_2,flow_t_minus_2,t_minus_1,flow_t_minus_1,t0,flow_t0,t_plus_1,flow_t_plus_1,t_plus_2,flow_t_plus_2,t_plus_3,flow_t_plus_3
0,INCIDENT__219427-webtirf__7120,7120,2025-01-01 04:00:00,2025-01-01 01:00:00,0.0,2025-01-01 02:00:00,0.0,2025-01-01 03:00:00,0.0,2025-01-01 04:00:00,0.0,2025-01-01 05:00:00,0.0,2025-01-01 06:00:00,0.0,2025-01-01 07:00:00,0.0
1,INCIDENT__219427-webtirf__7121,7121,2025-01-01 04:00:00,2025-01-01 01:00:00,0.0,2025-01-01 02:00:00,0.0,2025-01-01 03:00:00,0.0,2025-01-01 04:00:00,0.0,2025-01-01 05:00:00,0.0,2025-01-01 06:00:00,0.0,2025-01-01 07:00:00,0.0
2,INCIDENT__219440.0-webtirf__7168,7168,2025-01-01 11:00:00,2025-01-01 08:00:00,13.0,2025-01-01 09:00:00,4.0,2025-01-01 10:00:00,13.0,2025-01-01 11:00:00,10.0,2025-01-01 12:00:00,16.0,2025-01-01 13:00:00,12.0,2025-01-01 14:00:00,7.0
3,INCIDENT__219468-webtirf__F3FWY003,F3FWY003,2025-01-02 01:00:00,2025-01-01 22:00:00,0.0,2025-01-01 23:00:00,0.0,2025-01-02 00:00:00,31.0,2025-01-02 01:00:00,19.0,2025-01-02 02:00:00,30.0,2025-01-02 03:00:00,27.0,2025-01-02 04:00:00,0.0
4,INCIDENT__219481-webtirf__7179,7179,2025-01-02 03:00:00,2025-01-02 00:00:00,21.0,2025-01-02 01:00:00,30.0,2025-01-02 02:00:00,33.0,2025-01-02 03:00:00,34.0,2025-01-02 04:00:00,73.0,2025-01-02 05:00:00,105.0,2025-01-02 06:00:00,160.0


### A22 — Assertions

In [69]:
assert (
    valid_incident_dataset[
        FLOW_COLUMNS
    ]
    .notna()
    .all()
    .all()
)

assert valid_incident_dataset[
    "event_sample_id"
].is_unique

assert (
    len(
        valid_incident_dataset
    )
    ==
    valid_incident_windows_long[
        "event_sample_id"
    ].nunique()
)

assert (
    len(
        valid_incident_dataset
    )
    == 1788
)

print(
    "PASS: all retained incident samples "
    "contain complete 7-hour traffic windows."
)

PASS: all retained incident samples contain complete 7-hour traffic windows.


In [70]:
# check the class distribution:
display(
    valid_incident_dataset[
        "target_class"
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

,samples
target_class,
OTHER_DISRUPTION,1232
ACCIDENT,556


### A23 — Check conflicting incidents inside each event window

In [71]:
# Prepare event-level incident records
known_incident_events = (
    incident_samples[
        incident_samples[
            "target_class"
        ].isin(
            [
                "ACCIDENT",
                "OTHER_DISRUPTION",
            ]
        )
    ]
    [
        [
            "event_sample_id",
            "incident_id",
            "station_id",
            "anchor_time",
            "target_class",
        ]
    ]
    .copy()
)

print(
    "Known incident-station events:",
    len(known_incident_events)
)

display(
    known_incident_events.head()
)

Known incident-station events: 1788


,event_sample_id,incident_id,station_id,anchor_time,target_class
0,INCIDENT__219427-webtirf__7120,219427-webtirf,7120,2025-01-01 04:00:00,OTHER_DISRUPTION
1,INCIDENT__219427-webtirf__7121,219427-webtirf,7121,2025-01-01 04:00:00,OTHER_DISRUPTION
2,INCIDENT__219440.0-webtirf__7168,219440.0-webtirf,7168,2025-01-01 11:00:00,OTHER_DISRUPTION
3,INCIDENT__219468-webtirf__F3FWY003,219468-webtirf,F3FWY003,2025-01-02 01:00:00,ACCIDENT
4,INCIDENT__219481-webtirf__7179,219481-webtirf,7179,2025-01-02 03:00:00,OTHER_DISRUPTION


### A24 — Find incidents occurring inside every target window

In [72]:
window_conflict_candidates = (
    valid_incident_dataset[
        [
            "event_sample_id",
            "incident_id",
            "station_id",
            "anchor_time",
            "window_start",
            "window_end",
            "target_class",
        ]
    ]
    .rename(
        columns={
            "incident_id":
                "target_incident_id",

            "target_class":
                "target_class_main",
        }
    )
    .merge(
        known_incident_events.rename(
            columns={
                "event_sample_id":
                    "other_event_sample_id",

                "incident_id":
                    "other_incident_id",

                "anchor_time":
                    "other_anchor_time",

                "target_class":
                    "other_target_class",
            }
        ),
        on="station_id",
        how="left",
    )
)

### A25 — Identify real conflicts

In [74]:
conflicts = (
    window_conflict_candidates[
        (
            window_conflict_candidates[
                "other_incident_id"
            ]
            !=
            window_conflict_candidates[
                "target_incident_id"
            ]
        )
        &
        (
            window_conflict_candidates[
                "other_anchor_time"
            ]
            >=
            window_conflict_candidates[
                "window_start"
            ]
        )
        &
        (
            window_conflict_candidates[
                "other_anchor_time"
            ]
            <=
            window_conflict_candidates[
                "window_end"
            ]
        )
        &
        (
            window_conflict_candidates[
                "other_target_class"
            ]
            !=
            window_conflict_candidates[
                "target_class_main"
            ]
        )
    ]
    .copy()
)

In [75]:
print(
    "Conflicting incident pairs:",
    len(conflicts)
)

print(
    "Affected target samples:",
    conflicts[
        "event_sample_id"
    ].nunique()
)

Conflicting incident pairs: 34
Affected target samples: 34


In [76]:
display(
    conflicts[
        [
            "event_sample_id",
            "station_id",
            "target_incident_id",
            "target_class_main",
            "anchor_time",
            "window_start",
            "window_end",
            "other_incident_id",
            "other_target_class",
            "other_anchor_time",
        ]
    ]
    .sort_values(
        [
            "station_id",
            "anchor_time",
        ]
    )
    .head(30)
)

,event_sample_id,station_id,target_incident_id,target_class_main,anchor_time,window_start,window_end,other_incident_id,other_target_class,other_anchor_time
80517,INCIDENT__256664-webtirf__23067,23067,256664-webtirf,ACCIDENT,2025-11-23 19:00:00,2025-11-23 16:00:00,2025-11-23 22:00:00,256677-webtirf,OTHER_DISRUPTION,2025-11-23 20:00:00
80566,INCIDENT__256677-webtirf__23067,23067,256677-webtirf,OTHER_DISRUPTION,2025-11-23 20:00:00,2025-11-23 17:00:00,2025-11-23 23:00:00,256664-webtirf,ACCIDENT,2025-11-23 19:00:00
46278,INCIDENT__242317-webtirf__33014,33014,242317-webtirf,OTHER_DISRUPTION,2025-07-15 06:00:00,2025-07-15 03:00:00,2025-07-15 09:00:00,242323-webtirf,ACCIDENT,2025-07-15 07:00:00
46514,INCIDENT__242323-webtirf__33014,33014,242323-webtirf,ACCIDENT,2025-07-15 07:00:00,2025-07-15 04:00:00,2025-07-15 10:00:00,242317-webtirf,OTHER_DISRUPTION,2025-07-15 06:00:00
9515,INCIDENT__223195-webtirf__47024,47024,223195-webtirf,ACCIDENT,2025-02-07 22:00:00,2025-02-07 19:00:00,2025-02-08 01:00:00,223214-webtirf,OTHER_DISRUPTION,2025-02-08 01:00:00
9588,INCIDENT__223214-webtirf__47024,47024,223214-webtirf,OTHER_DISRUPTION,2025-02-08 01:00:00,2025-02-07 22:00:00,2025-02-08 04:00:00,223195-webtirf,ACCIDENT,2025-02-07 22:00:00
88662,INCIDENT__259505-webtirf__50240,50240,259505-webtirf,OTHER_DISRUPTION,2025-12-17 06:00:00,2025-12-17 03:00:00,2025-12-17 09:00:00,259509-webtirf,ACCIDENT,2025-12-17 06:00:00
88722,INCIDENT__259509-webtirf__50240,50240,259509-webtirf,ACCIDENT,2025-12-17 06:00:00,2025-12-17 03:00:00,2025-12-17 09:00:00,259505-webtirf,OTHER_DISRUPTION,2025-12-17 06:00:00
43011,INCIDENT__241192-webtirf__51235,51235,241192-webtirf,OTHER_DISRUPTION,2025-07-03 05:00:00,2025-07-03 02:00:00,2025-07-03 08:00:00,241193-webtirf,ACCIDENT,2025-07-03 05:00:00
43062,INCIDENT__241193-webtirf__51235,51235,241193-webtirf,ACCIDENT,2025-07-03 05:00:00,2025-07-03 02:00:00,2025-07-03 08:00:00,241192-webtirf,OTHER_DISRUPTION,2025-07-03 05:00:00


### A26 — Exclude conflicting samples

In [77]:
conflicting_sample_ids = set(
    conflicts[
        "event_sample_id"
    ].unique()
)

clean_incident_dataset = (
    valid_incident_dataset[
        ~valid_incident_dataset[
            "event_sample_id"
        ].isin(
            conflicting_sample_ids
        )
    ]
    .copy()
)

In [78]:
# check how many remain
print(
    "Incident samples before conflict exclusion:",
    len(valid_incident_dataset)
)

print(
    "Excluded conflicting samples:",
    len(conflicting_sample_ids)
)

print(
    "Incident samples after conflict exclusion:",
    len(clean_incident_dataset)
)

Incident samples before conflict exclusion: 1788
Excluded conflicting samples: 34
Incident samples after conflict exclusion: 1754


In [79]:
# Check class distribution
display(
    clean_incident_dataset[
        "target_class"
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

,samples
target_class,
OTHER_DISRUPTION,1215
ACCIDENT,539


### A27 — Add the exclusion to the audit table

In [80]:
incident_window_audit[
    "conflicting_target_class"
] = (
    incident_window_audit[
        "event_sample_id"
    ].isin(
        conflicting_sample_ids
    )
)

In [81]:
# Create the final inclusion status
incident_window_audit[
    "final_included"
] = (
    incident_window_audit[
        "window_valid"
    ]
    &
    ~incident_window_audit[
        "conflicting_target_class"
    ]
)


In [82]:
display(
    incident_window_audit[
        [
            "window_valid",
            "conflicting_target_class",
            "final_included",
        ]
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

samples
window_valid conflicting_target_class final_included         
True         False                    True               1754
             True                     False                34

### A28 — Audit all-zero traffic windows

In [83]:
clean_incident_dataset[
    "all_zero_flow_window"
] = (
    clean_incident_dataset[
        FLOW_COLUMNS
    ]
    .eq(0)
    .all(
        axis=1
    )
)

In [84]:
print(
    "All-zero 7-hour windows:",
    clean_incident_dataset[
        "all_zero_flow_window"
    ].sum()
)

print(
    "Percentage:",
    round(
        clean_incident_dataset[
            "all_zero_flow_window"
        ].mean()
        * 100,
        2,
    ),
    "%",
)

All-zero 7-hour windows: 423
Percentage: 24.12 %


In [85]:
# by class:
display(
    pd.crosstab(
        clean_incident_dataset[
            "target_class"
        ],
        clean_incident_dataset[
            "all_zero_flow_window"
        ],
        margins=True,
    )
)

all_zero_flow_window,False,True,All
target_class,,,
ACCIDENT,420,119,539
OTHER_DISRUPTION,911,304,1215
All,1331,423,1754


In [86]:
# by station:
display(
    clean_incident_dataset[
        clean_incident_dataset[
            "all_zero_flow_window"
        ]
    ]
    .groupby(
        "station_id"
    )
    .size()
    .sort_values(
        ascending=False
    )
    .to_frame(
        "all_zero_windows"
    )
    .head(20)
)

,all_zero_windows
station_id,
10011,41
33014,35
7121,33
F3FWY003,26
7120,23
7112,17
T0289,14
22001,14
T0288,13


### A29 — Audit all-zero stations

In [87]:
station_flow_audit = (
    flow_df
    .groupby("station_id")
    .agg(
        total_observations=("total_flow", "size"),
        zero_observations=(
            "total_flow",
            lambda x: (x == 0).sum(),
        ),
        nonzero_observations=(
            "total_flow",
            lambda x: (x > 0).sum(),
        ),
        mean_flow=("total_flow", "mean"),
        median_flow=("total_flow", "median"),
        max_flow=("total_flow", "max"),
    )
    .reset_index()
)

station_flow_audit[
    "zero_percentage"
] = (
    station_flow_audit[
        "zero_observations"
    ]
    /
    station_flow_audit[
        "total_observations"
    ]
    * 100
)

In [88]:
# Inspect stations responsible for all-zero incident windows:

In [89]:
all_zero_station_ids = (
    clean_incident_dataset.loc[
        clean_incident_dataset[
            "all_zero_flow_window"
        ],
        "station_id",
    ]
    .unique()
)

display(
    station_flow_audit[
        station_flow_audit[
            "station_id"
        ].isin(
            all_zero_station_ids
        )
    ]
    .sort_values(
        "zero_percentage",
        ascending=False,
    )
    .head(30)
)

,station_id,total_observations,zero_observations,nonzero_observations,mean_flow,median_flow,max_flow,zero_percentage
5,32029,8760,8284,476,22.499658,0.0,1912.0,94.566210
1,10011,8760,7599,1161,82.531735,0.0,3185.0,86.746575
2,22001,8760,7570,1190,56.476826,0.0,1786.0,86.415525
98,T0289,8760,7518,1242,14.600799,0.0,372.0,85.821918
97,T0288,8760,7268,1492,20.624087,0.0,481.0,82.968037
6,33014,8760,5378,3382,172.473402,0.0,2129.0,61.392694
99,T0290,8760,5315,3445,43.295662,0.0,436.0,60.673516
83,F3FWY006,8760,4788,3972,279.619064,0.0,3330.0,54.657534
80,F3FWY003,8760,4775,3985,377.159247,0.0,4387.0,54.509132
82,F3FWY005,8760,2823,5937,513.464269,162.0,4319.0,32.226027


### A30 — Identify permanently or mostly-zero stations

In [90]:
display(
    station_flow_audit[
        [
            "station_id",
            "total_observations",
            "zero_observations",
            "nonzero_observations",
            "zero_percentage",
            "mean_flow",
            "median_flow",
            "max_flow",
        ]
    ]
    .sort_values(
        "zero_percentage",
        ascending=False,
    )
    .head(30)
)

,station_id,total_observations,zero_observations,nonzero_observations,zero_percentage,mean_flow,median_flow,max_flow
102,T0296,8760,8706,54,99.383562,0.057192,0.0,37.0
22,6114,8760,8415,345,96.061644,1.969064,0.0,350.0
19,6109,8760,8361,399,95.445205,7.323174,0.0,932.0
5,32029,8760,8284,476,94.566210,22.499658,0.0,1912.0
20,6110,8760,8203,557,93.641553,7.902511,0.0,935.0
21,6113,8760,7892,868,90.091324,5.786644,0.0,375.0
1,10011,8760,7599,1161,86.746575,82.531735,0.0,3185.0
2,22001,8760,7570,1190,86.415525,56.476826,0.0,1786.0
98,T0289,8760,7518,1242,85.821918,14.600799,0.0,372.0
97,T0288,8760,7268,1492,82.968037,20.624087,0.0,481.0


In [91]:
station_flow_audit[
    "flow_status"
] = pd.cut(
    station_flow_audit[
        "zero_percentage"
    ],
    bins=[
        -0.1,
        50,
        90,
        99,
        100.1,
    ],
    labels=[
        "MOSTLY_ACTIVE",
        "FREQUENT_ZEROS",
        "MOSTLY_ZERO",
        "ALMOST_ALWAYS_ZERO",
    ],
)

In [92]:
display(
    station_flow_audit[
        "flow_status"
    ]
    .value_counts()
    .to_frame(
        "stations"
    )
)

,stations
flow_status,
MOSTLY_ACTIVE,93
FREQUENT_ZEROS,16
MOSTLY_ZERO,5
ALMOST_ALWAYS_ZERO,1


### A31 — Check whether zero windows occur at station inactive periods

In [93]:
zero_window_station_summary = (
    clean_incident_dataset[
        clean_incident_dataset[
            "all_zero_flow_window"
        ]
    ]
    .groupby("station_id")
    .size()
    .reset_index(
        name="all_zero_incident_windows"
    )
    .merge(
        station_flow_audit,
        on="station_id",
        how="left",
    )
    .sort_values(
        "all_zero_incident_windows",
        ascending=False,
    )
)

display(
    zero_window_station_summary.head(30)
)

,station_id,all_zero_incident_windows,total_observations,zero_observations,nonzero_observations,mean_flow,median_flow,max_flow,zero_percentage,flow_status
0,10011,41,8760,7599,1161,82.531735,0.0,3185.0,86.746575,FREQUENT_ZEROS
5,33014,35,8760,5378,3382,172.473402,0.0,2129.0,61.392694,FREQUENT_ZEROS
19,7121,33,8760,2620,6140,112.982991,21.0,709.0,29.908676,MOSTLY_ACTIVE
47,F3FWY003,26,8760,4775,3985,377.159247,0.0,4387.0,54.509132,FREQUENT_ZEROS
18,7120,23,8760,1844,6916,203.605365,50.5,861.0,21.050228,MOSTLY_ACTIVE
16,7112,17,8760,2101,6659,324.785502,87.0,1782.0,23.984018,MOSTLY_ACTIVE
53,T0289,14,8760,7518,1242,14.600799,0.0,372.0,85.821918,FREQUENT_ZEROS
1,22001,14,8760,7570,1190,56.476826,0.0,1786.0,86.415525,FREQUENT_ZEROS
52,T0288,13,8760,7268,1492,20.624087,0.0,481.0,82.968037,FREQUENT_ZEROS
31,7179,12,8760,1364,7396,564.802055,253.5,2339.0,15.570776,MOSTLY_ACTIVE


### A32 — Check all-zero windows by hour of day

In [94]:
clean_incident_dataset[
    "anchor_hour"
] = (
    clean_incident_dataset[
        "anchor_time"
    ].dt.hour
)

zero_by_hour = pd.crosstab(
    clean_incident_dataset[
        "anchor_hour"
    ],
    clean_incident_dataset[
        "all_zero_flow_window"
    ],
)

zero_by_hour.columns = [
    "nonzero_window",
    "all_zero_window",
]

zero_by_hour[
    "zero_window_percentage"
] = (
    zero_by_hour[
        "all_zero_window"
    ]
    /
    zero_by_hour.sum(
        axis=1
    )
    * 100
)

display(
    zero_by_hour
)

,nonzero_window,all_zero_window,zero_window_percentage
anchor_hour,,,
0,80,21,20.792079
1,73,14,16.091954
2,75,24,24.242424
3,90,39,30.232558
4,100,30,23.076923
5,104,40,27.777778
6,100,32,24.242424
7,77,24,23.762376
8,64,16,20.000000


### A33 — Construct split_group_id

In [104]:
# Sort samples chronologically by station
grouping_dataset[
    "split_group_id"
] = (
    "GROUP__"
    + grouping_dataset[
        "station_id"
    ].astype(str)
    + "__"
    + grouping_dataset[
        "station_group_number"
    ]
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

### A34 — Create leakage groups within each station

In [105]:
def assign_station_split_groups(station_group):
    """
    Assign connected temporal leakage groups within one station.

    Two event samples belong to the same group when their
    7-hour windows overlap.

    Because overlapping relationships are transitive,
    the current group end is extended whenever a newly
    connected window reaches further into the future.
    """

    station_group = (
        station_group
        .sort_values(
            [
                "window_start",
                "window_end",
                "anchor_time",
            ]
        )
        .copy()
    )

    local_group_numbers = []

    current_group = 0
    current_group_end = None

    for _, row in station_group.iterrows():

        if current_group_end is None:

            current_group += 1
            current_group_end = row["window_end"]

        elif row["window_start"] <= current_group_end:

            # Connected to the existing component
            current_group_end = max(
                current_group_end,
                row["window_end"],
            )

        else:

            # Start a new independent component
            current_group += 1
            current_group_end = row["window_end"]

        local_group_numbers.append(
            current_group
        )

    station_group[
        "station_group_number"
    ] = local_group_numbers

    return station_group

In [106]:
grouping_dataset = (
    grouping_dataset
    .groupby(
        "station_id",
        group_keys=False,
    )
    .apply(
        assign_station_split_groups
    )
    .reset_index(
        drop=True
    )
)

### A35 — Create globally unique split_group_id

In [107]:
display(
    grouping_dataset[
        [
            "event_sample_id",
            "station_id",
            "anchor_time",
            "window_start",
            "window_end",
            "split_group_id",
        ]
    ].head(30)
)

,event_sample_id,station_id,anchor_time,window_start,window_end,split_group_id
0,INCIDENT__240340.0-webtirf__100001,100001,2025-06-27 00:00:00,2025-06-26 21:00:00,2025-06-27 03:00:00,GROUP__100001__0001
1,INCIDENT__247639-webtirf__100001,100001,2025-08-29 05:00:00,2025-08-29 02:00:00,2025-08-29 08:00:00,GROUP__100001__0002
2,INCIDENT__250888-webtirf__100001,100001,2025-09-27 13:00:00,2025-09-27 10:00:00,2025-09-27 16:00:00,GROUP__100001__0003
3,INCIDENT__219957-webtirf__10011,10011,2025-01-08 19:00:00,2025-01-08 16:00:00,2025-01-08 22:00:00,GROUP__10011__0001
4,INCIDENT__221139-webtirf__10011,10011,2025-01-17 21:00:00,2025-01-17 18:00:00,2025-01-18 00:00:00,GROUP__10011__0002
5,INCIDENT__221259-webtirf__10011,10011,2025-01-18 23:00:00,2025-01-18 20:00:00,2025-01-19 02:00:00,GROUP__10011__0003
6,INCIDENT__222066-webtirf__10011,10011,2025-01-28 05:00:00,2025-01-28 02:00:00,2025-01-28 08:00:00,GROUP__10011__0004
7,INCIDENT__223065-webtirf__10011,10011,2025-02-06 20:00:00,2025-02-06 17:00:00,2025-02-06 23:00:00,GROUP__10011__0005
8,INCIDENT__223348-webtirf__10011,10011,2025-02-09 21:00:00,2025-02-09 18:00:00,2025-02-10 00:00:00,GROUP__10011__0006
9,INCIDENT__225542-webtirf__10011,10011,2025-03-03 19:00:00,2025-03-03 16:00:00,2025-03-03 22:00:00,GROUP__10011__0007


### A36 — Audit the groups

In [108]:
split_group_summary = (
    grouping_dataset
    .groupby(
        "split_group_id"
    )
    .agg(
        station_id=(
            "station_id",
            "first",
        ),

        group_start=(
            "window_start",
            "min",
        ),

        group_end=(
            "window_end",
            "max",
        ),

        sample_count=(
            "event_sample_id",
            "size",
        ),

        class_count=(
            "target_class",
            "nunique",
        ),
    )
    .reset_index()
)

In [109]:
print(
    "Total incident samples:",
    len(grouping_dataset)
)

print(
    "Total split groups:",
    grouping_dataset[
        "split_group_id"
    ].nunique()
)

print(
    "Largest split group:",
    split_group_summary[
        "sample_count"
    ].max()
)

print(
    "Groups containing >1 sample:",
    (
        split_group_summary[
            "sample_count"
        ] > 1
    ).sum()
)

Total incident samples: 1754
Total split groups: 1616
Largest split group: 4
Groups containing >1 sample: 129


In [110]:
display(
    split_group_summary[
        "sample_count"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "groups"
    )
)

,groups
sample_count,
1,1487
2,121
3,7
4,1


### A37 — Inspect the largest groups

In [111]:
largest_group_ids = (
    split_group_summary
    .sort_values(
        "sample_count",
        ascending=False,
    )
    .head(10)[
        "split_group_id"
    ]
)

display(
    grouping_dataset[
        grouping_dataset[
            "split_group_id"
        ].isin(
            largest_group_ids
        )
    ]
    [
        [
            "split_group_id",
            "event_sample_id",
            "station_id",
            "anchor_time",
            "target_class",
            "window_start",
            "window_end",
        ]
    ]
    .sort_values(
        [
            "split_group_id",
            "anchor_time",
        ]
    )
)

,split_group_id,event_sample_id,station_id,anchor_time,target_class,window_start,window_end
162,GROUP__33014__0008,INCIDENT__225994-webtirf__33014,33014,2025-03-06 06:00:00,OTHER_DISRUPTION,2025-03-06 03:00:00,2025-03-06 09:00:00
163,GROUP__33014__0008,INCIDENT__225995-webtirf__33014,33014,2025-03-06 06:00:00,OTHER_DISRUPTION,2025-03-06 03:00:00,2025-03-06 09:00:00
548,GROUP__7112__0015,INCIDENT__227288-webtirf__7112,7112,2025-03-12 18:00:00,ACCIDENT,2025-03-12 15:00:00,2025-03-12 21:00:00
549,GROUP__7112__0015,INCIDENT__227316-webtirf__7112,7112,2025-03-12 23:00:00,OTHER_DISRUPTION,2025-03-12 20:00:00,2025-03-13 02:00:00
550,GROUP__7112__0015,INCIDENT__227351-webtirf__7112,7112,2025-03-13 02:00:00,OTHER_DISRUPTION,2025-03-12 23:00:00,2025-03-13 05:00:00
591,GROUP__7112__0052,INCIDENT__249800-webtirf__7112,7112,2025-09-17 00:00:00,OTHER_DISRUPTION,2025-09-16 21:00:00,2025-09-17 03:00:00
592,GROUP__7112__0052,INCIDENT__249800.0-webtirf__7112,7112,2025-09-17 00:00:00,OTHER_DISRUPTION,2025-09-16 21:00:00,2025-09-17 03:00:00
593,GROUP__7112__0052,INCIDENT__249840-webtirf__7112,7112,2025-09-17 05:00:00,OTHER_DISRUPTION,2025-09-17 02:00:00,2025-09-17 08:00:00
594,GROUP__7112__0052,INCIDENT__249840.0-webtirf__7112,7112,2025-09-17 05:00:00,OTHER_DISRUPTION,2025-09-17 02:00:00,2025-09-17 08:00:00
709,GROUP__7120__0061,INCIDENT__255350-webtirf__7120,7120,2025-11-11 03:00:00,OTHER_DISRUPTION,2025-11-11 00:00:00,2025-11-11 06:00:00


### A38 — Important leakage assertion

In [112]:
def check_grouping_leakage(df):

    violations = []

    for station_id, station_df in df.groupby(
        "station_id"
    ):

        station_df = (
            station_df
            .sort_values(
                "window_start"
            )
            .reset_index(
                drop=True
            )
        )

        for i in range(
            len(station_df)
        ):

            row_i = (
                station_df.iloc[i]
            )

            for j in range(
                i + 1,
                len(station_df),
            ):

                row_j = (
                    station_df.iloc[j]
                )

                # Later windows can no longer overlap
                if (
                    row_j[
                        "window_start"
                    ]
                    >
                    row_i[
                        "window_end"
                    ]
                ):
                    break

                overlap = (
                    row_i[
                        "window_start"
                    ]
                    <=
                    row_j[
                        "window_end"
                    ]
                ) and (
                    row_j[
                        "window_start"
                    ]
                    <=
                    row_i[
                        "window_end"
                    ]
                )

                if (
                    overlap
                    and
                    row_i[
                        "split_group_id"
                    ]
                    !=
                    row_j[
                        "split_group_id"
                    ]
                ):

                    violations.append(
                        {
                            "station_id":
                                station_id,

                            "event_1":
                                row_i[
                                    "event_sample_id"
                                ],

                            "event_2":
                                row_j[
                                    "event_sample_id"
                                ],
                        }
                    )

    return pd.DataFrame(
        violations
    )

In [117]:
assert grouping_dataset["split_group_id"].notna().all()

assert (
    grouping_dataset
    .groupby("split_group_id")["station_id"]
    .nunique()
    .max()
    == 1
)

print("PASS: split groups are complete and station-specific.")

PASS: split groups are complete and station-specific.


In [113]:
grouping_violations = (
    check_grouping_leakage(
        grouping_dataset
    )
)

print(
    "Grouping leakage violations:",
    len(
        grouping_violations
    )
)

Grouping leakage violations: 0


### A39 — Rename this as the working incident dataset

In [115]:
incident_dataset_grouped = (
    grouping_dataset
    .copy()
)

## Phase B — Create NORMAL controls

### B1 — Build the incident exclusion calendar

In [ ]:
# B1.1 Create all incident anchors
all_incident_events = (
    incident_df
    .groupby(
        [
            "incident_id",
            "station_id",
        ],
        as_index=False,
    )
    .agg(
        incident_anchor=(
            "match_hour",
            "min",
        )
    )
)

print(
    "All incident-station events:",
    len(all_incident_events)
)

display(
    all_incident_events.head()
)

All incident-station events: 1828


,incident_id,station_id,incident_anchor
0,219427-webtirf,7120,2025-01-01 04:00:00
1,219427-webtirf,7121,2025-01-01 04:00:00
2,219440.0-webtirf,7168,2025-01-01 11:00:00
3,219468-webtirf,F3FWY003,2025-01-02 01:00:00
4,219481-webtirf,7179,2025-01-02 03:00:00


In [119]:
# B1.2 — Prepare event windows that controls cannot overlap
all_incident_events[
    "event_window_start"
] = (
    all_incident_events[
        "incident_anchor"
    ]
    - pd.Timedelta(
        hours=3
    )
)

all_incident_events[
    "event_window_end"
] = (
    all_incident_events[
        "incident_anchor"
    ]
    + pd.Timedelta(
        hours=3
    )
)

In [120]:
display(
    all_incident_events.head()
)

,incident_id,station_id,incident_anchor,event_window_start,event_window_end
0,219427-webtirf,7120,2025-01-01 04:00:00,2025-01-01 01:00:00,2025-01-01 07:00:00
1,219427-webtirf,7121,2025-01-01 04:00:00,2025-01-01 01:00:00,2025-01-01 07:00:00
2,219440.0-webtirf,7168,2025-01-01 11:00:00,2025-01-01 08:00:00,2025-01-01 14:00:00
3,219468-webtirf,F3FWY003,2025-01-02 01:00:00,2025-01-01 22:00:00,2025-01-02 04:00:00
4,219481-webtirf,7179,2025-01-02 03:00:00,2025-01-02 00:00:00,2025-01-02 06:00:00


### B2 — Generate candidate control anchors

In [121]:
# B2.1 — Prepare available hourly anchors
control_anchor_pool = (
    flow_df[
        [
            "station_id",
            "timestamp",
        ]
    ]
    .copy()
    .drop_duplicates()
)

control_anchor_pool[
    "hour_of_day"
] = (
    control_anchor_pool[
        "timestamp"
    ].dt.hour
)

control_anchor_pool[
    "day_of_week"
] = (
    control_anchor_pool[
        "timestamp"
    ].dt.dayofweek
)

control_anchor_pool[
    "is_weekend_match"
] = (
    control_anchor_pool[
        "day_of_week"
    ]
    >= 5
).astype(int)

print(
    "Candidate hourly anchors:",
    len(control_anchor_pool)
)

Candidate hourly anchors: 1007400


In [122]:
# B2.2 — Add matching fields to incident samples
incident_dataset_grouped[
    "anchor_hour"
] = (
    incident_dataset_grouped[
        "anchor_time"
    ].dt.hour
)

incident_dataset_grouped[
    "anchor_day_of_week"
] = (
    incident_dataset_grouped[
        "anchor_time"
    ].dt.dayofweek
)

incident_dataset_grouped[
    "anchor_is_weekend"
] = (
    incident_dataset_grouped[
        "anchor_day_of_week"
    ]
    >= 5
).astype(int)

In [123]:
display(
    incident_dataset_grouped[
        [
            "event_sample_id",
            "station_id",
            "anchor_time",
            "anchor_hour",
            "anchor_day_of_week",
            "anchor_is_weekend",
        ]
    ].head()
)

,event_sample_id,station_id,anchor_time,anchor_hour,anchor_day_of_week,anchor_is_weekend
0,INCIDENT__240340.0-webtirf__100001,100001,2025-06-27 00:00:00,0,4,0
1,INCIDENT__247639-webtirf__100001,100001,2025-08-29 05:00:00,5,4,0
2,INCIDENT__250888-webtirf__100001,100001,2025-09-27 13:00:00,13,5,1
3,INCIDENT__219957-webtirf__10011,10011,2025-01-08 19:00:00,19,2,0
4,INCIDENT__221139-webtirf__10011,10011,2025-01-17 21:00:00,21,4,0


### B3 — Control selection

In [124]:
# B3.1 — Build fast lookup structures
incident_times_by_station = (
    all_incident_events
    .groupby(
        "station_id"
    )[
        "incident_anchor"
    ]
    .apply(list)
    .to_dict()
)


event_windows_by_station = (
    all_incident_events
    .groupby(
        "station_id"
    )[
        [
            "event_window_start",
            "event_window_end",
        ]
    ]
    .apply(
        lambda x: list(
            x.itertuples(
                index=False,
                name=None,
            )
        )
    )
    .to_dict()
)

In [125]:
# Build control candidates by station:
control_pool_by_station = {
    station_id: group.copy()
    for station_id, group
    in control_anchor_pool.groupby(
        "station_id"
    )
}

In [126]:
# B3.2 — Define the validity test
def is_valid_control_anchor(
    station_id,
    control_anchor,
):
    """
    Check whether a candidate control anchor is incident-free.
    """

    # ------------------------------------------
    # Rule 1:
    # No incident of any type within ±7 hours
    # ------------------------------------------

    incident_times = (
        incident_times_by_station
        .get(
            station_id,
            []
        )
    )

    for incident_time in incident_times:

        time_difference = abs(
            (
                control_anchor
                - incident_time
            ).total_seconds()
            / 3600
        )

        if time_difference <= 7:
            return False

    # ------------------------------------------
    # Rule 2:
    # Control 7-hour window cannot overlap
    # any event window
    # ------------------------------------------

    control_start = (
        control_anchor
        - pd.Timedelta(
            hours=3
        )
    )

    control_end = (
        control_anchor
        + pd.Timedelta(
            hours=3
        )
    )

    event_windows = (
        event_windows_by_station
        .get(
            station_id,
            []
        )
    )

    for (
        event_start,
        event_end,
    ) in event_windows:

        overlap = (
            control_start
            <= event_end
        ) and (
            event_start
            <= control_end
        )

        if overlap:
            return False

    return True

In [127]:
# B3.3 — Select up to two controls
CONTROL_RANDOM_SEED = 42

control_rng = np.random.default_rng(
    CONTROL_RANDOM_SEED
)

used_control_anchors = set()

selected_controls = []

In [128]:
# the selection loop:
for _, incident_row in (
    incident_dataset_grouped
    .sort_values(
        "event_sample_id"
    )
    .iterrows()
):

    station_id = (
        incident_row[
            "station_id"
        ]
    )

    incident_anchor = (
        incident_row[
            "anchor_time"
        ]
    )

    hour_of_day = (
        incident_row[
            "anchor_hour"
        ]
    )

    day_of_week = (
        incident_row[
            "anchor_day_of_week"
        ]
    )

    is_weekend = (
        incident_row[
            "anchor_is_weekend"
        ]
    )

    station_pool = (
        control_pool_by_station.get(
            station_id
        )
    )

    if station_pool is None:
        continue

    # ------------------------------------------
    # Exact day-of-week candidates
    # ------------------------------------------

    exact_candidates = (
        station_pool[
            (
                station_pool[
                    "hour_of_day"
                ]
                == hour_of_day
            )
            &
            (
                station_pool[
                    "day_of_week"
                ]
                == day_of_week
            )
            &
            (
                station_pool[
                    "timestamp"
                ].dt.date
                !=
                incident_anchor.date()
            )
        ]
        .copy()
    )

    valid_exact = []

    for candidate_anchor in (
        exact_candidates[
            "timestamp"
        ]
    ):

        candidate_key = (
            station_id,
            candidate_anchor,
        )

        if (
            candidate_key
            in used_control_anchors
        ):
            continue

        if is_valid_control_anchor(
            station_id,
            candidate_anchor,
        ):

            valid_exact.append(
                candidate_anchor
            )

    # ------------------------------------------
    # Use exact day-of-week where available
    # ------------------------------------------

    if len(valid_exact) > 0:

        candidate_list = (
            valid_exact
        )

        fallback_flag = 0

    else:

        # --------------------------------------
        # Weekday/weekend fallback
        # --------------------------------------

        fallback_candidates = (
            station_pool[
                (
                    station_pool[
                        "hour_of_day"
                    ]
                    == hour_of_day
                )
                &
                (
                    station_pool[
                        "is_weekend_match"
                    ]
                    == is_weekend
                )
                &
                (
                    station_pool[
                        "timestamp"
                    ].dt.date
                    !=
                    incident_anchor.date()
                )
            ]
            .copy()
        )

        candidate_list = []

        for candidate_anchor in (
            fallback_candidates[
                "timestamp"
            ]
        ):

            candidate_key = (
                station_id,
                candidate_anchor,
            )

            if (
                candidate_key
                in used_control_anchors
            ):
                continue

            if is_valid_control_anchor(
                station_id,
                candidate_anchor,
            ):

                candidate_list.append(
                    candidate_anchor
                )

        fallback_flag = 1

    # ------------------------------------------
    # Randomize valid candidates deterministically
    # ------------------------------------------

    candidate_list = list(
        candidate_list
    )

    control_rng.shuffle(
        candidate_list
    )

    selected_for_incident = (
        candidate_list[:2]
    )

    # ------------------------------------------
    # Save selected controls
    # ------------------------------------------

    for control_number, control_anchor in enumerate(
        selected_for_incident,
        start=1,
    ):

        control_key = (
            station_id,
            control_anchor,
        )

        used_control_anchors.add(
            control_key
        )

        selected_controls.append(
            {
                "source_event_sample_id":
                    incident_row[
                        "event_sample_id"
                    ],

                "source_incident_id":
                    incident_row[
                        "incident_id"
                    ],

                "source_split_group_id":
                    incident_row[
                        "split_group_id"
                    ],

                "station_id":
                    station_id,

                "control_anchor":
                    control_anchor,

                "control_number":
                    control_number,

                "fallback_flag":
                    fallback_flag,

                "target_class":
                    "NORMAL",
            }
        )

In [129]:
# B3.4 — Create the control table
control_samples = pd.DataFrame(
    selected_controls
)

control_samples[
    "event_sample_id"
] = (
    "CONTROL__"
    +
    control_samples[
        "station_id"
    ].astype(str)
    +
    "__"
    +
    control_samples[
        "control_anchor"
    ].dt.strftime(
        "%Y%m%d%H"
    )
)

In [130]:
print(
    "Selected NORMAL controls:",
    len(control_samples)
)

print(
    "Incident samples:",
    len(
        incident_dataset_grouped
    )
)

print(
    "Controls per incident ratio:",
    round(
        len(control_samples)
        /
        len(
            incident_dataset_grouped
        ),
        3,
    )
)

Selected NORMAL controls: 3508
Incident samples: 1754
Controls per incident ratio: 2.0


In [131]:
# Check fallback usage:
display(
    control_samples[
        "fallback_flag"
    ]
    .value_counts()
    .rename(
        index={
            0:
                "Exact day-of-week",

            1:
                "Weekday/weekend fallback",
        }
    )
    .to_frame(
        "controls"
    )
)

,controls
fallback_flag,
Exact day-of-week,3508


In [132]:
# Check how many controls each incident received:
controls_per_incident = (
    control_samples
    .groupby(
        "source_event_sample_id"
    )
    .size()
)

display(
    controls_per_incident
    .value_counts()
    .sort_index()
    .rename_axis(
        "controls_received"
    )
    .to_frame(
        "incidents"
    )
)

,incidents
controls_received,
2,1754


In [133]:
# check incidents receiving zero controls:
incidents_with_controls = set(
    control_samples[
        "source_event_sample_id"
    ]
)

zero_control_incidents = (
    incident_dataset_grouped[
        ~incident_dataset_grouped[
            "event_sample_id"
        ].isin(
            incidents_with_controls
        )
    ]
)

print(
    "Incidents receiving zero controls:",
    len(
        zero_control_incidents
    )
)

Incidents receiving zero controls: 0


### Phase B4 — Extract 7-hour NORMAL control windows

In [134]:
# B4.1 — Create control window timestamps
CONTROL_TIME_OFFSETS = {
    "t_minus_3": -3,
    "t_minus_2": -2,
    "t_minus_1": -1,
    "t0": 0,
    "t_plus_1": 1,
    "t_plus_2": 2,
    "t_plus_3": 3,
}


control_window_rows = []

for timestep_name, offset in CONTROL_TIME_OFFSETS.items():

    temp = control_samples[
        [
            "event_sample_id",
            "source_event_sample_id",
            "source_incident_id",
            "source_split_group_id",
            "station_id",
            "control_anchor",
            "control_number",
            "fallback_flag",
            "target_class",
        ]
    ].copy()

    temp["timestep"] = timestep_name
    temp["relative_hour"] = offset

    temp["timestamp"] = (
        temp["control_anchor"]
        + pd.to_timedelta(
            offset,
            unit="h",
        )
    )

    control_window_rows.append(temp)


control_windows_long = pd.concat(
    control_window_rows,
    ignore_index=True,
)

print(
    "Control window rows:",
    len(control_windows_long)
)

Control window rows: 24556


In [135]:
# B4.2 — Join traffic flow
control_windows_long = (
    control_windows_long
    .merge(
        flow_lookup,
        on=[
            "station_id",
            "timestamp",
        ],
        how="left",
        validate="many_to_one",
    )
)

In [136]:
# Check missing flow:
control_missing_summary = (
    control_windows_long
    .groupby(
        "event_sample_id"
    )["total_flow"]
    .agg(
        missing_count=lambda x: x.isna().sum(),
        available_count=lambda x: x.notna().sum(),
    )
    .reset_index()
)

In [137]:
display(
    control_missing_summary[
        "missing_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "missing_hours"
    )
    .to_frame(
        "controls"
    )
)

,controls
missing_hours,
0,3505
1,2
3,1


In [138]:
# B4.2a — Inspect the incomplete controls
incomplete_control_ids = (
    control_missing_summary.loc[
        control_missing_summary["missing_count"] > 0,
        "event_sample_id",
    ]
)

display(
    control_windows_long[
        control_windows_long[
            "event_sample_id"
        ].isin(
            incomplete_control_ids
        )
    ]
    .sort_values(
        [
            "event_sample_id",
            "relative_hour",
        ]
    )
)

,event_sample_id,source_event_sample_id,source_incident_id,source_split_group_id,station_id,control_anchor,control_number,fallback_flag,target_class,timestep,relative_hour,timestamp,total_flow
2779,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_minus_3,-3,2024-12-31 23:00:00,NaN
6287,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_minus_2,-2,2025-01-01 00:00:00,1667.0
9795,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_minus_1,-1,2025-01-01 01:00:00,2120.0
13303,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t0,0,2025-01-01 02:00:00,1814.0
16811,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_plus_1,1,2025-01-01 03:00:00,952.0
20319,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_plus_2,2,2025-01-01 04:00:00,560.0
23827,CONTROL__7139__2025010102,INCIDENT__254102-webtirf__7139,254102-webtirf,GROUP__7139__0038,7139,2025-01-01 02:00:00,2,0,NORMAL,t_plus_3,3,2025-01-01 05:00:00,389.0
815,CONTROL__7161__2025010102,INCIDENT__228703-webtirf__7161,228703-webtirf,GROUP__7161__0008,7161,2025-01-01 02:00:00,2,0,NORMAL,t_minus_3,-3,2024-12-31 23:00:00,NaN
4323,CONTROL__7161__2025010102,INCIDENT__228703-webtirf__7161,228703-webtirf,GROUP__7161__0008,7161,2025-01-01 02:00:00,2,0,NORMAL,t_minus_2,-2,2025-01-01 00:00:00,332.0
7831,CONTROL__7161__2025010102,INCIDENT__228703-webtirf__7161,228703-webtirf,GROUP__7161__0008,7161,2025-01-01 02:00:00,2,0,NORMAL,t_minus_1,-1,2025-01-01 01:00:00,339.0


In [139]:
# B4.2b — Apply the same interpolation function
control_windows_processed = (
    control_windows_long
    .groupby(
        "event_sample_id",
        group_keys=False,
    )
    .apply(
        interpolate_incident_window
    )
    .reset_index(
        drop=True
    )
)

In [141]:
# create the audit:
control_window_audit = (
    control_windows_processed
    .groupby(
        "event_sample_id"
    )
    .agg(
        window_valid=(
            "window_valid",
            "all",
        ),

        interpolated_points=(
            "flow_was_interpolated",
            "sum",
        ),

        exclusion_reason=(
            "exclusion_reason",
            "first",
        ),

        final_missing_count=(
            "total_flow",
            lambda x: x.isna().sum(),
        ),
    )
    .reset_index()
)

In [142]:
print("Control window validity:")

display(
    control_window_audit[
        "window_valid"
    ]
    .value_counts()
    .to_frame(
        "controls"
    )
)

print("Interpolated points:")

display(
    control_window_audit[
        "interpolated_points"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "controls"
    )
)

print("Exclusion reasons:")

display(
    control_window_audit[
        "exclusion_reason"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "controls"
    )
)

Control window validity:


,controls
window_valid,
True,3505
False,3


Interpolated points:


,controls
interpolated_points,
0,3508


Exclusion reasons:


,controls
exclusion_reason,
None,3505
MISSING_FLOW_CANNOT_BE_INTERPOLATED,2
MORE_THAN_ONE_MISSING_FLOW,1


In [143]:
# B4.2c — Keep only valid controls
valid_control_ids = set(
    control_window_audit.loc[
        control_window_audit[
            "window_valid"
        ],
        "event_sample_id",
    ]
)

control_windows_valid_long = (
    control_windows_processed[
        control_windows_processed[
            "event_sample_id"
        ].isin(
            valid_control_ids
        )
    ]
    .copy()
)

control_samples_valid = (
    control_samples[
        control_samples[
            "event_sample_id"
        ].isin(
            valid_control_ids
        )
    ]
    .copy()
)

In [144]:
print(
    "Original controls:",
    len(control_samples)
)

print(
    "Valid controls after window check:",
    len(control_samples_valid)
)

print(
    "Dropped controls:",
    len(control_samples)
    - len(control_samples_valid)
)

Original controls: 3508
Valid controls after window check: 3505
Dropped controls: 3


In [145]:
# B4.3 — Convert valid controls to wide format
control_flow_wide = (
    control_windows_valid_long
    .pivot(
        index="event_sample_id",
        columns="timestep",
        values="total_flow",
    )
    .reset_index()
)

control_flow_wide.columns.name = None

control_flow_wide = (
    control_flow_wide
    .rename(
        columns={
            "t_minus_3": "flow_t_minus_3",
            "t_minus_2": "flow_t_minus_2",
            "t_minus_1": "flow_t_minus_1",
            "t0": "flow_t0",
            "t_plus_1": "flow_t_plus_1",
            "t_plus_2": "flow_t_plus_2",
            "t_plus_3": "flow_t_plus_3",
        }
    )
)

In [148]:
# build the final control dataset:
control_dataset = (
    control_samples_valid
    .merge(
        control_flow_wide,
        on="event_sample_id",
        how="inner",
        validate="one_to_one",
    )
)

In [149]:
# Add the common anchor and window fields:
control_dataset["anchor_time"] = (
    control_dataset["control_anchor"]
)

control_dataset["window_start"] = (
    control_dataset["anchor_time"]
    - pd.Timedelta(hours=3)
)

control_dataset["window_end"] = (
    control_dataset["anchor_time"]
    + pd.Timedelta(hours=3)
)

In [151]:
print(
    "Incident samples:",
    len(incident_dataset_grouped)
)

print(
    "NORMAL samples:",
    len(control_dataset)
)

print(
    "Total samples:",
    len(incident_dataset_grouped)
    + len(control_dataset)
)

Incident samples: 1754
NORMAL samples: 3505
Total samples: 5259


### Phase C — Build the canonical modelling dataset.


In [ ]:
# C1 — Align incident and NORMAL samples
control_dataset[
    "split_group_id"
] = control_dataset[
    "source_split_group_id"
]

In [154]:
# For consistency, add a sample-type column:
incident_dataset_grouped[
    "sample_type"
] = "INCIDENT"

control_dataset[
    "sample_type"
] = "CONTROL"

In [155]:
# C2 — Select common modelling columns
MODEL_COLUMNS = [
    "event_sample_id",
    "station_id",
    "anchor_time",
    "window_start",
    "window_end",
    "target_class",
    "split_group_id",
    "sample_type",
    "flow_t_minus_3",
    "flow_t_minus_2",
    "flow_t_minus_1",
    "flow_t0",
    "flow_t_plus_1",
    "flow_t_plus_2",
    "flow_t_plus_3",
]

In [156]:
# combine
canonical_dataset = pd.concat(
    [
        incident_dataset_grouped[
            MODEL_COLUMNS
        ],
        control_dataset[
            MODEL_COLUMNS
        ],
    ],
    ignore_index=True,
)

In [157]:
# C3 — Validate the final dataset
print(
    "Canonical dataset shape:",
    canonical_dataset.shape
)

display(
    canonical_dataset[
        "target_class"
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

Canonical dataset shape: (5259, 15)


,samples
target_class,
NORMAL,3505
OTHER_DISRUPTION,1215
ACCIDENT,539


In [158]:
# Check sample IDs:
print(
    "Duplicate sample IDs:",
    canonical_dataset[
        "event_sample_id"
    ].duplicated().sum()
)

Duplicate sample IDs: 0


In [160]:
# check missing flow:
print(
    "Samples with missing flow:",
    canonical_dataset[
        FLOW_COLUMNS
    ]
    .isna()
    .any(axis=1)
    .sum()
)

Samples with missing flow: 0


In [161]:
# Check classes:
assert set(
    canonical_dataset[
        "target_class"
    ].unique()
) == {
    "NORMAL",
    "ACCIDENT",
    "OTHER_DISRUPTION",
}

In [162]:
# final assertions:
assert len(
    canonical_dataset
) == 5259

assert canonical_dataset[
    "event_sample_id"
].is_unique

assert canonical_dataset[
    "split_group_id"
].notna().all()

assert canonical_dataset[
    FLOW_COLUMNS
].notna().all().all()

print(
    "PASS: canonical 3-class dataset is ready."
)

PASS: canonical 3-class dataset is ready.


### Phase D — Create the frozen train/validation/test splits

### D1 — Grouped-random split

In [187]:
# D1.1 — Summarise each split group
group_summary = (
    canonical_dataset
    .groupby("split_group_id")
    .agg(
        sample_count=(
            "event_sample_id",
            "size",
        ),
        normal_count=(
            "target_class",
            lambda x: (x == "NORMAL").sum(),
        ),
        accident_count=(
            "target_class",
            lambda x: (x == "ACCIDENT").sum(),
        ),
        other_count=(
            "target_class",
            lambda x: (x == "OTHER_DISRUPTION").sum(),
        ),
    )
    .reset_index()
)

print(
    "Total split groups:",
    len(group_summary)
)

display(
    group_summary.head()
)

Total split groups: 1616


,split_group_id,sample_count,normal_count,accident_count,other_count
0,GROUP__100001__0001,3,2,1,0
1,GROUP__100001__0002,3,2,1,0
2,GROUP__100001__0003,3,2,1,0
3,GROUP__10011__0001,3,2,1,0
4,GROUP__10011__0002,3,2,0,1


In [191]:
from sklearn.model_selection import train_test_split

In [192]:
group_summary["stratify_label"] = np.select(
    [
        group_summary["accident_count"] > 0,
        group_summary["other_count"] > 0,
    ],
    [
        "ACCIDENT_GROUP",
        "OTHER_GROUP",
    ],
    default="NORMAL_GROUP",
)

display(
    group_summary[
        "stratify_label"
    ].value_counts()
)

stratify_label
OTHER_GROUP       1103
ACCIDENT_GROUP     513
Name: count, dtype: int64

In [193]:
train_groups, temp_groups = train_test_split(
    group_summary,
    test_size=0.30,
    random_state=42,
    stratify=group_summary[
        "stratify_label"
    ],
)

In [194]:
validation_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=42,
    stratify=temp_groups[
        "stratify_label"
    ],
)

In [195]:
# create assignment
group_assignments = pd.concat(
    [
        train_groups[
            ["split_group_id"]
        ].assign(
            split="train"
        ),

        validation_groups[
            ["split_group_id"]
        ].assign(
            split="validation"
        ),

        test_groups[
            ["split_group_id"]
        ].assign(
            split="test"
        ),
    ],
    ignore_index=True,
)

In [196]:
grouped_random_manifest = (
    canonical_dataset
    .merge(
        group_assignments,
        on="split_group_id",
        how="left",
        validate="many_to_one",
    )
)

### D2 — Validate the grouped-random split

In [197]:
# D2.1 — Split sizes
grouped_split_counts = (
    grouped_random_manifest[
        "split"
    ]
    .value_counts()
)

display(
    grouped_split_counts
    .to_frame(
        "samples"
    )
)

,samples
split,
train,3695
test,788
validation,776


In [198]:
# show percentages
grouped_split_percentages = (
    grouped_split_counts
    /
    len(
        grouped_random_manifest
    )
    * 100
)

display(
    grouped_split_percentages
    .round(2)
    .to_frame(
        "percentage"
    )
)

,percentage
split,
train,70.26
test,14.98
validation,14.76


In [199]:
# D2.2 — Class distribution by split
grouped_class_distribution = (
    pd.crosstab(
        grouped_random_manifest[
            "split"
        ],
        grouped_random_manifest[
            "target_class"
        ],
    )
)

display(
    grouped_class_distribution
)

target_class,ACCIDENT,NORMAL,OTHER_DISRUPTION
split,,,
test,81,525,182
train,379,2463,853
validation,79,517,180


In [200]:
# check row percentages:
display(
    (
        grouped_class_distribution
        .div(
            grouped_class_distribution.sum(
                axis=1
            ),
            axis=0,
        )
        * 100
    )
    .round(2)
)

target_class,ACCIDENT,NORMAL,OTHER_DISRUPTION
split,,,
test,10.28,66.62,23.10
train,10.26,66.66,23.09
validation,10.18,66.62,23.20


In [203]:
# D2.3 — Leakage assertion
group_leakage = (
    grouped_random_manifest
    .groupby(
        "split_group_id"
    )["split"]
    .nunique()
)

print(
    "Groups appearing in multiple splits:",
    (group_leakage > 1).sum()
)

assert group_leakage.max() == 1

print(
    "PASS: grouped-random split has no split-group leakage."
)

Groups appearing in multiple splits: 0
PASS: grouped-random split has no split-group leakage.


In [202]:
assert (
    group_leakage
    .max()
    == 1
)

print(
    "PASS: grouped-random split "
    "has no split-group leakage."
)

PASS: grouped-random split has no split-group leakage.


### D3 — Create the unseen-station spatial split

In [176]:
# D3.1 — Summarise samples by station
station_summary = (
    canonical_dataset
    .groupby(
        "station_id"
    )
    .agg(
        sample_count=(
            "event_sample_id",
            "size",
        ),

        normal_count=(
            "target_class",
            lambda x:
                (x == "NORMAL").sum(),
        ),

        accident_count=(
            "target_class",
            lambda x:
                (x == "ACCIDENT").sum(),
        ),

        other_count=(
            "target_class",
            lambda x:
                (
                    x
                    ==
                    "OTHER_DISRUPTION"
                ).sum(),
        ),
    )
    .reset_index()
)

print(
    "Total stations:",
    len(station_summary)
)

Total stations: 67


In [177]:
# D3.2 — Search for a good spatial assignment
def evaluate_spatial_assignment(
    assignment_df,
    station_summary,
):

    merged = (
        station_summary
        .merge(
            assignment_df,
            on="station_id",
            how="left",
        )
    )

    summary = (
        merged
        .groupby(
            "split"
        )[
            [
                "sample_count",
                "normal_count",
                "accident_count",
                "other_count",
            ]
        ]
        .sum()
    )

    desired_ratios = {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    }

    total = (
        station_summary[
            [
                "sample_count",
                "normal_count",
                "accident_count",
                "other_count",
            ]
        ]
        .sum()
    )

    score = 0

    for split_name, ratio in (
        desired_ratios.items()
    ):

        if split_name not in summary.index:
            return np.inf

        for column in total.index:

            target = (
                total[column]
                * ratio
            )

            score += (
                (
                    summary.loc[
                        split_name,
                        column,
                    ]
                    - target
                )
                /
                max(
                    target,
                    1,
                )
            ) ** 2

    # Support requirements
    for split_name in [
        "validation",
        "test",
    ]:

        if (
            summary.loc[
                split_name,
                "normal_count",
            ]
            < 10
        ):
            return np.inf

        if (
            summary.loc[
                split_name,
                "accident_count",
            ]
            < 10
        ):
            return np.inf

        if (
            summary.loc[
                split_name,
                "other_count",
            ]
            < 10
        ):
            return np.inf

    return score

In [178]:
# search up to 50 assignments:
best_spatial_assignment = None
best_spatial_score = np.inf

station_ids = (
    station_summary[
        "station_id"
    ]
    .tolist()
)

for seed in range(50):

    rng = np.random.default_rng(
        seed
    )

    shuffled_stations = (
        station_ids.copy()
    )

    rng.shuffle(
        shuffled_stations
    )

    n_stations = len(
        shuffled_stations
    )

    train_end = int(
        round(
            n_stations
            * 0.70
        )
    )

    val_end = (
        train_end
        +
        int(
            round(
                n_stations
                * 0.15
            )
        )
    )

    assignment = []

    for i, station_id in enumerate(
        shuffled_stations
    ):

        if i < train_end:

            split_name = (
                "train"
            )

        elif i < val_end:

            split_name = (
                "validation"
            )

        else:

            split_name = (
                "test"
            )

        assignment.append(
            {
                "station_id":
                    station_id,

                "split":
                    split_name,
            }
        )

    assignment_df = (
        pd.DataFrame(
            assignment
        )
    )

    score = (
        evaluate_spatial_assignment(
            assignment_df,
            station_summary,
        )
    )

    if score < best_spatial_score:

        best_spatial_score = score

        best_spatial_assignment = (
            assignment_df.copy()
        )

In [179]:
# Check that one was found:
assert (
    best_spatial_assignment
    is not None
)

print(
    "Best spatial score:",
    best_spatial_score
)

Best spatial score: 0.02294455441300053


In [180]:
# D3.3 — Build spatial manifest
spatial_manifest = (
    canonical_dataset
    .merge(
        best_spatial_assignment,
        on="station_id",
        how="left",
        validate="many_to_one",
    )
)

### D4 — Validate the spatial split

In [181]:
# D4.1 — Split sizes
display(
    spatial_manifest[
        "split"
    ]
    .value_counts()
    .to_frame(
        "samples"
    )
)

,samples
split,
train,3700
validation,797
test,762


In [182]:
# percentages:
display(
    (
        spatial_manifest[
            "split"
        ]
        .value_counts(
            normalize=True
        )
        * 100
    )
    .round(2)
    .to_frame(
        "percentage"
    )
)

,percentage
split,
train,70.36
validation,15.15
test,14.49


In [183]:
# D4.2 — Class support
spatial_class_distribution = (
    pd.crosstab(
        spatial_manifest[
            "split"
        ],
        spatial_manifest[
            "target_class"
        ],
    )
)

display(
    spatial_class_distribution
)

target_class,ACCIDENT,NORMAL,OTHER_DISRUPTION
split,,,
test,78,508,176
train,370,2466,864
validation,91,531,175


In [204]:
# D4.3 — Station counts
display(
    spatial_manifest
    .groupby(
        "split"
    )[
        "station_id"
    ]
    .nunique()
    .to_frame(
        "stations"
    )
)

,stations
split,
test,10
train,47
validation,10


In [205]:
# D4.4 — Station leakage assertion
station_leakage = (
    spatial_manifest
    .groupby(
        "station_id"
    )[
        "split"
    ]
    .nunique()
)

print(
    "Stations appearing in multiple splits:",
    (
        station_leakage > 1
    ).sum()
)

Stations appearing in multiple splits: 0


In [206]:
assert (
    station_leakage.max()
    == 1
)

print(
    "PASS: spatial split "
    "has no station leakage."
)

PASS: spatial split has no station leakage.


## Phase E — Export the frozen benchmark files

In [209]:
# Compact grouped-random split manifest
grouped_random_split_manifest = (
    grouped_random_manifest[
        [
            "event_sample_id",
            "split_group_id",
            "station_id",
            "target_class",
            "split",
        ]
    ]
    .copy()
)

# Compact spatial split manifest
spatial_split_manifest = (
    spatial_manifest[
        [
            "event_sample_id",
            "split_group_id",
            "station_id",
            "target_class",
            "split",
        ]
    ]
    .copy()
)

print(
    "Grouped-random manifest:",
    grouped_random_split_manifest.shape
)

print(
    "Spatial manifest:",
    spatial_split_manifest.shape
)

Grouped-random manifest: (5259, 5)
Spatial manifest: (5259, 5)


In [211]:
canonical_dataset.to_csv(
    OUTPUT_DIR / "canonical_dataset.csv",
    index=False,
)

grouped_random_split_manifest.to_csv(
    OUTPUT_DIR / "grouped_random_split_manifest.csv",
    index=False,
)

spatial_split_manifest.to_csv(
    OUTPUT_DIR / "spatial_split_manifest.csv",
    index=False,
)

print("Benchmark files exported successfully.")

Benchmark files exported successfully.


## Data Preparation Complete

The final classification benchmark contains 5,259 samples across three classes:

- NORMAL: 3,505
- OTHER_DISRUPTION: 1,215
- ACCIDENT: 539

Two frozen evaluation protocols are provided:

1. Grouped-random split (70/15/15), preventing overlapping event groups from crossing partitions.
2. Spatial split, preventing stations from appearing across train, validation, and test partitions.

The exported datasets are used as fixed inputs for all classification models.